# Multi-Object Measurement Notebook
Detects and measures **multiple objects** in one image — including **rotated rectangles and squares** — and overlays dimensions in real-world units.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import requests


## Scale Calibration
Set the known reference: how many pixels correspond to a known real-world length.

In [ ]:
# Known scale: 10 cm = 535 pixels  (update for your setup)
PIXELS_PER_10CM = 535
CM_PER_PIXEL = 10.0 / PIXELS_PER_10CM
CM2_PER_PIXEL2 = CM_PER_PIXEL ** 2

print(f"Scale: 1 pixel = {CM_PER_PIXEL:.5f} cm")


## Load Image

In [ ]:
image_url = "https://raw.githubusercontent.com/ChiriKamau/notebooks/main/plank_test.jpg"

response = requests.get(image_url)
image_array = np.array(bytearray(response.content), dtype=np.uint8)
image = cv2.imdecode(image_array, cv2.IMREAD_COLOR)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8, 8))
plt.imshow(image_rgb)
plt.title("Original Image")
plt.axis("off")
plt.show()


## Preprocessing & Segmentation

In [ ]:
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (7, 7), 0)

# Otsu threshold
_, binary = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Make sure objects are white
if np.mean(binary) > 127:
    binary = cv2.bitwise_not(binary)

# Morphological cleanup
kernel = np.ones((5, 5), np.uint8)
binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

plt.figure(figsize=(7, 7))
plt.imshow(binary, cmap="gray")
plt.title("Cleaned Binary Mask")
plt.axis("off")
plt.show()


## Multi-Object Detection & Measurement

### Key improvements over the original:
- **All objects** above a minimum area threshold are detected, not just the largest one.
- **Oriented (rotated) bounding box** via `cv2.minAreaRect` accurately measures width and height of rotated rectangles/squares, regardless of their angle.
- A shape classifier labels each object as **Square**, **Rectangle**, or **Other** based on the aspect ratio of the oriented box.
- Results are annotated directly on the image.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#   TUNABLE PARAMETERS  — adjust these to your image
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MIN_AREA_PX2          = 500   # minimum contour area in pixels² (noise filter)
SQUARE_RATIO_TOLERANCE = 0.10  # if short/long side ≥ 0.90  →  Square

# ── Find & filter contours ─────────────────────────────────────────────
contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
contours     = [c for c in contours if cv2.contourArea(c) > MIN_AREA_PX2]

print(f"\n{'━'*40}")
print(f"  Objects detected: {len(contours)}")
print(f"{'━'*40}\n")

# ── Measure every object ───────────────────────────────────────────────
results = []

for idx, contour in enumerate(contours, start=1):

    # Area & Perimeter
    area_px  = cv2.contourArea(contour)
    area_cm2 = area_px * CM2_PER_PIXEL2
    peri_px  = cv2.arcLength(contour, True)
    peri_cm  = peri_px * CM_PER_PIXEL

    # Oriented (rotated) bounding box  →  true W × H for tilted objects
    rect              = cv2.minAreaRect(contour)
    centre, (bw, bh), angle = rect

    # Normalise: always make bw = long side
    if bw < bh:
        bw, bh = bh, bw
        angle  = angle + 90

    width_cm  = bw * CM_PER_PIXEL
    height_cm = bh * CM_PER_PIXEL

    # Shape classification
    ratio = min(bw, bh) / max(bw, bh) if max(bw, bh) > 0 else 1
    if ratio >= (1 - SQUARE_RATIO_TOLERANCE):
        shape = "Square"
    elif ratio >= 0.3:
        shape = "Rectangle"
    else:
        shape = "Other"

    results.append(dict(
        id=idx, shape=shape,
        area_cm2=area_cm2, peri_cm=peri_cm,
        width_cm=width_cm, height_cm=height_cm,
        angle_deg=angle, centre=centre, rect=rect,
    ))

print("Measurements complete — run the next cell to visualise.")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#   VISUALISATION  — annotated output image
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Colour palette per shape
COLOURS = {
    "Square"    : (  0, 220, 255),   # cyan
    "Rectangle" : (255, 100,   0),   # orange
    "Other"     : (180,   0, 255),   # purple
}
TEXT_COLOUR  = (255, 255,  60)       # bright yellow
BOX_THICK    = 3
FONT_SCALE   = 1.2                   # ← bigger text
FONT_THICK   = 2
LINE_GAP     = 42                    # pixels between annotation lines

output = image_rgb.copy()

for r in results:
    contour = contours[r["id"] - 1]
    colour  = COLOURS.get(r["shape"], (200, 200, 200))
    rect    = r["rect"]
    cx, cy  = int(r["centre"][0]), int(r["centre"][1])

    # Semi-transparent object fill
    overlay = output.copy()
    cv2.drawContours(overlay, [contour], -1, colour, -1)
    output  = cv2.addWeighted(overlay, 0.20, output, 0.80, 0)

    # Oriented bounding box
    box_pts = cv2.boxPoints(rect).astype(np.int32)
    cv2.drawContours(output, [box_pts], -1, colour, BOX_THICK)

    # Object ID badge (filled circle)
    badge_radius = 22
    cv2.circle(output, (cx, cy), badge_radius, colour, -1)
    cv2.circle(output, (cx, cy), badge_radius, (255,255,255), 2)
    id_txt = str(r["id"])
    (tw, th), _ = cv2.getTextSize(id_txt, cv2.FONT_HERSHEY_DUPLEX, 0.8, 2)
    cv2.putText(output, id_txt,
                (cx - tw//2, cy + th//2),
                cv2.FONT_HERSHEY_DUPLEX, 0.8, (20, 20, 20), 2, cv2.LINE_AA)

    # Annotation lines (top-right of object)
    ax = cx + badge_radius + 10
    ay = cy - LINE_GAP

    lines = [
        f"#{r['id']}  {r['shape']}",
        f"{r['width_cm']:.1f} x {r['height_cm']:.1f} cm",
        f"Area: {r['area_cm2']:.1f} cm²",
        f"Angle: {r['angle_deg']:.1f}°",
    ]

    for i, line in enumerate(lines):
        # Shadow
        cv2.putText(output, line,
                    (ax + 2, ay + i * LINE_GAP + 2),
                    cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE,
                    (0, 0, 0), FONT_THICK + 2, cv2.LINE_AA)
        # Main text
        cv2.putText(output, line,
                    (ax, ay + i * LINE_GAP),
                    cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE,
                    TEXT_COLOUR, FONT_THICK, cv2.LINE_AA)

# ── Show ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 16))
ax.imshow(output)
ax.set_title(
    f"Detected {len(results)} object(s)  ·  Oriented Bounding Boxes",
    fontsize=18, fontweight="bold", pad=14
)
ax.axis("off")
plt.tight_layout()
plt.show()


## Summary Table

In [ ]:
print(f"{'ID':<4} {'Shape':<12} {'Width(cm)':<12} {'Height(cm)':<12} "
      f"{'Area(cm2)':<12} {'Perim(cm)':<12} {'Angle(deg)':<12}")
print("-" * 76)
for r in results:
    print(f"{r['id']:<4} {r['shape']:<12} {r['width_cm']:<12.2f} {r['height_cm']:<12.2f} "
          f"{r['area_cm2']:<12.2f} {r['peri_cm']:<12.2f} {r['angle_deg']:<12.1f}")
